# Zomato Delivery Operations Analysis

This notebook analyzes the Zomato delivery dataset to cover five areas:

1. **Delivery Performance Analysis** – ratings, delivery times, vehicle condition
2. **Route Optimization** – distance vs. traffic density vs. delivery time
3. **Customer Experience** – weather & festival impact on delivery time
4. **Operational Insights** – order volumes, order types, multiple deliveries
5. **Predictive Analytics** – ML models to forecast delivery time

**How to use in Google Colab:**
1. Upload `Zomato_Dataset.csv` when prompted in the cell below (or place it in your Drive and adjust the path).
2. Run cells top to bottom (`Runtime > Run all`).


## 1. Setup & Data Loading

In [ ]:
# Install/import required libraries (most are pre-installed on Colab)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", None)


In [ ]:
# --- Load the dataset ---
# Option A: Upload directly in Colab
try:
    from google.colab import files
    uploaded = files.upload()  # choose Zomato_Dataset.csv when prompted
    filename = list(uploaded.keys())[0]
except ImportError:
    # Not running in Colab (e.g. local Jupyter) -- edit path as needed
    filename = "Zomato_Dataset.csv"

df = pd.read_csv(filename)
print("Shape:", df.shape)
df.head()


## 2. Data Cleaning & Feature Engineering

The raw file has a few quirks we need to handle before analysis:
- Missing values in `Delivery_person_Age`, `Delivery_person_Ratings`, `Time_Orderd`, `Weather_conditions`, `Road_traffic_density`, `multiple_deliveries`, `Festival`, `City`
- `Time_taken (min)` needs a cleaner column name
- We derive **trip distance** from restaurant/delivery lat-long using the Haversine formula
- We derive **order hour**, **day of week**, and a **prep time** (gap between order placed and order picked up)


In [ ]:
df = df.copy()
df.columns = [c.strip() for c in df.columns]
df.rename(columns={"Time_taken (min)": "Time_taken_min"}, inplace=True)

# Strip whitespace from string/object columns
str_cols = df.select_dtypes(include="object").columns
for c in str_cols:
    df[c] = df[c].astype(str).str.strip().replace({"nan": np.nan, "NaN": np.nan})

# Parse dates/times
df["Order_Date"] = pd.to_datetime(df["Order_Date"], format="%d-%m-%Y", errors="coerce")
df["Time_Orderd_parsed"] = pd.to_datetime(df["Time_Orderd"], format="%H:%M", errors="coerce")
df["Time_Order_picked_parsed"] = pd.to_datetime(df["Time_Order_picked"], format="%H:%M", errors="coerce")

df["Order_Hour"] = df["Time_Orderd_parsed"].dt.hour
df["Order_DayOfWeek"] = df["Order_Date"].dt.day_name()
df["Order_Month"] = df["Order_Date"].dt.month_name()

# Prep time in minutes (handles orders that cross midnight)
prep = (df["Time_Order_picked_parsed"] - df["Time_Orderd_parsed"]).dt.total_seconds() / 60
prep = prep.apply(lambda x: x + 24*60 if pd.notnull(x) and x < 0 else x)
df["Prep_time_min"] = prep

# Haversine distance between restaurant and delivery location (km)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df["Distance_km"] = haversine(
    df["Restaurant_latitude"], df["Restaurant_longitude"],
    df["Delivery_location_latitude"], df["Delivery_location_longitude"]
)

# Some lat/long pairs in this dataset are stored as near-zero or negative placeholders -- clip obvious junk
df.loc[df["Distance_km"] > 50, "Distance_km"] = np.nan
df.loc[df["Distance_km"] < 0.01, "Distance_km"] = np.nan

df["multiple_deliveries"] = pd.to_numeric(df["multiple_deliveries"], errors="coerce")

print(df.isnull().sum().sort_values(ascending=False).head(12))
df[["Order_Hour","Order_DayOfWeek","Prep_time_min","Distance_km"]].describe(include="all")


## 3. Delivery Performance Analysis

Looking at how delivery-person ratings, vehicle condition, and delivery time relate to each other.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["Delivery_person_Ratings"].dropna(), bins=20, kde=True, ax=axes[0], color="teal")
axes[0].set_title("Distribution of Delivery Person Ratings")

sns.histplot(df["Time_taken_min"], bins=30, kde=True, ax=axes[1], color="orange")
axes[1].set_title("Distribution of Delivery Time (min)")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df.sample(min(5000, len(df)), random_state=1),
                 x="Delivery_person_Ratings", y="Time_taken_min", alpha=0.3)
sns.regplot(data=df, x="Delivery_person_Ratings", y="Time_taken_min",
            scatter=False, color="red", line_kws={"linewidth":2})
plt.title("Delivery Person Rating vs Delivery Time")
plt.show()

print("Correlation (rating vs time):",
      df[["Delivery_person_Ratings","Time_taken_min"]].corr().iloc[0,1].round(3))


In [ ]:
plt.figure(figsize=(7,5))
sns.boxplot(data=df, x="Vehicle_condition", y="Time_taken_min", palette="viridis")
plt.title("Delivery Time by Vehicle Condition (0=worst, 3=best)")
plt.show()

print(df.groupby("Vehicle_condition")["Time_taken_min"].agg(["mean","median","count"]))


In [ ]:
# Top / bottom performing delivery persons (by average rating and average delivery time)
perf = df.groupby("Delivery_person_ID").agg(
    avg_rating=("Delivery_person_Ratings", "mean"),
    avg_time=("Time_taken_min", "mean"),
    orders=("ID", "count")
).query("orders >= 5")

print("Top 10 rated delivery persons:")
print(perf.sort_values("avg_rating", ascending=False).head(10))

print("\nSlowest 10 delivery persons (avg time, min 5 orders):")
print(perf.sort_values("avg_time", ascending=False).head(10))


**Takeaways to look for:** a mild negative correlation between rating and delivery time (better-rated riders tend to deliver faster), and vehicle condition 0 (poor) usually shows the highest average delivery time -- a clear maintenance/replacement priority.

## 4. Route Optimization

Relating trip distance and traffic density to delivery time to see where the biggest time losses happen.


In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df.sample(min(5000, len(df)), random_state=1),
                 x="Distance_km", y="Time_taken_min", alpha=0.3, hue="Road_traffic_density")
plt.title("Distance vs Delivery Time, colored by Traffic Density")
plt.show()


In [ ]:
order = ["Low","Medium","High","Jam"]
present_order = [o for o in order if o in df["Road_traffic_density"].dropna().unique()]

plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="Road_traffic_density", y="Time_taken_min", order=present_order, palette="Reds")
plt.title("Delivery Time by Road Traffic Density")
plt.show()

print(df.groupby("Road_traffic_density")["Time_taken_min"].mean().reindex(present_order))


In [ ]:
# Average delivery speed (km per minute) by traffic density -- proxy for route efficiency
df["Speed_km_per_min"] = df["Distance_km"] / df["Time_taken_min"]
speed_by_traffic = df.groupby("Road_traffic_density")["Speed_km_per_min"].mean().reindex(present_order)
print(speed_by_traffic)

speed_by_traffic.plot(kind="bar", color="slateblue", title="Avg Effective Speed by Traffic Density")
plt.ylabel("km per minute")
plt.show()


In [ ]:
# City type vs delivery time -- helps prioritize which city types need route/staffing optimization
plt.figure(figsize=(7,5))
sns.boxplot(data=df, x="City", y="Time_taken_min", palette="crest")
plt.title("Delivery Time by City Type")
plt.show()


**Takeaways to look for:** delivery time rises sharply from Low -> Jam traffic density even after controlling for distance, and effective speed (km/min) drops in heavy traffic -- these are the routes/time-windows where re-routing or staggered dispatch would help most. Semi-urban/urban vs metropolitan comparisons flag city types needing more delivery partners.

## 5. Customer Experience Enhancement

Exploring how weather and festival season affect delivery time (a proxy for customer wait-time experience).


In [ ]:
plt.figure(figsize=(9,5))
weather_order = df.groupby("Weather_conditions")["Time_taken_min"].mean().sort_values().index
sns.boxplot(data=df, x="Weather_conditions", y="Time_taken_min", order=weather_order, palette="Blues")
plt.title("Delivery Time by Weather Condition")
plt.xticks(rotation=20)
plt.show()

print(df.groupby("Weather_conditions")["Time_taken_min"].agg(["mean","median","count"]).sort_values("mean"))


In [ ]:
plt.figure(figsize=(6,5))
sns.boxplot(data=df, x="Festival", y="Time_taken_min", palette="Set2")
plt.title("Delivery Time: Festival vs Non-Festival Days")
plt.show()

print(df.groupby("Festival")["Time_taken_min"].agg(["mean","median","count"]))


In [ ]:
# Combined view: festival x traffic density
pivot = df.pivot_table(index="Road_traffic_density", columns="Festival",
                        values="Time_taken_min", aggfunc="mean").reindex(present_order)
print(pivot)

pivot.plot(kind="bar", figsize=(8,5), title="Avg Delivery Time: Traffic Density x Festival")
plt.ylabel("Time taken (min)")
plt.show()


**Takeaways to look for:** Fog/Stormy/Sandstorm conditions and festival days both push delivery time up noticeably -- customers ordering under these conditions should get proactive ETA adjustments or comms rather than a flat promised time.

## 6. Operational Insights

Order volumes over time, order-type mix, and the effect of multiple deliveries per trip.


In [ ]:
plt.figure(figsize=(10,5))
df["Order_Date"].value_counts().sort_index().plot(kind="line", marker="o", markersize=3)
plt.title("Order Volume Over Time")
plt.ylabel("Number of Orders")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

df["Type_of_order"].value_counts().plot(kind="bar", ax=axes[0], color="coral")
axes[0].set_title("Order Volume by Order Type")

df["Type_of_vehicle"].value_counts().plot(kind="bar", ax=axes[1], color="seagreen")
axes[1].set_title("Order Volume by Vehicle Type")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7,5))
sns.boxplot(data=df, x="multiple_deliveries", y="Time_taken_min", palette="mako")
plt.title("Delivery Time by Number of Multiple Deliveries per Trip")
plt.show()

print(df.groupby("multiple_deliveries")["Time_taken_min"].agg(["mean","count"]))


In [ ]:
# Order volume by hour of day -- useful for staffing/dispatch planning
plt.figure(figsize=(9,5))
df["Order_Hour"].value_counts().sort_index().plot(kind="bar", color="darkorange")
plt.title("Order Volume by Hour of Day")
plt.xlabel("Hour")
plt.ylabel("Number of Orders")
plt.show()


**Takeaways to look for:** trips bundling more simultaneous deliveries take longer on average -- worth weighing against the throughput gains. Peak order hours indicate when extra riders should be scheduled.

## 7. Predictive Analytics: Forecasting Delivery Time

We build regression models using weather, traffic density, distance, vehicle type/condition, multiple deliveries, festival, city, and order hour as predictors of `Time_taken_min`.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model_df = df.dropna(subset=[
    "Time_taken_min","Distance_km","Delivery_person_Ratings","Delivery_person_Age",
    "Weather_conditions","Road_traffic_density","multiple_deliveries","Festival","City"
]).copy()

numeric_features = ["Distance_km","Delivery_person_Age","Delivery_person_Ratings",
                     "multiple_deliveries","Vehicle_condition","Order_Hour","Prep_time_min"]
categorical_features = ["Weather_conditions","Road_traffic_density","Type_of_order",
                         "Type_of_vehicle","Festival","City"]

# Fill any remaining engineered-feature NaNs (e.g. Prep_time_min, Order_Hour) with median
for c in ["Order_Hour","Prep_time_min"]:
    model_df[c] = model_df[c].fillna(model_df[c].median())

X = model_df[numeric_features + categorical_features]
y = model_df["Time_taken_min"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

print("Training rows:", X_train.shape[0], " | Test rows:", X_test.shape[0])


In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

results = {}
for name, model in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    results[name] = {
        "MAE": mean_absolute_error(y_test, preds),
        "RMSE": mean_squared_error(y_test, preds) ** 0.5,
        "R2": r2_score(y_test, preds),
        "pipe": pipe
    }

results_df = pd.DataFrame({k: {m: v for m, v in r.items() if m != "pipe"} for k, r in results.items()}).T
print(results_df.sort_values("RMSE"))


In [ ]:
best_name = results_df.sort_values("RMSE").index[0]
best_pipe = results[best_name]["pipe"]
print(f"Best model: {best_name}")

# Feature importance (tree-based models only)
if hasattr(best_pipe.named_steps["model"], "feature_importances_"):
    feature_names = (numeric_features +
                      list(best_pipe.named_steps["prep"].named_transformers_["cat"]
                           .get_feature_names_out(categorical_features)))
    importances = best_pipe.named_steps["model"].feature_importances_
    imp_df = pd.DataFrame({"feature": feature_names, "importance": importances}) \
                .sort_values("importance", ascending=False).head(15)

    plt.figure(figsize=(8,6))
    sns.barplot(data=imp_df, x="importance", y="feature", palette="flare")
    plt.title(f"Top 15 Feature Importances ({best_name})")
    plt.tight_layout()
    plt.show()


In [ ]:
# Actual vs Predicted plot for the best model
preds = best_pipe.predict(X_test)
plt.figure(figsize=(7,7))
plt.scatter(y_test, preds, alpha=0.3)
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, "r--")
plt.xlabel("Actual Time Taken (min)")
plt.ylabel("Predicted Time Taken (min)")
plt.title(f"Actual vs Predicted Delivery Time ({best_name})")
plt.show()


### Using the model to forecast a new delivery

Example: predict delivery time for a hypothetical order given its conditions.


In [ ]:
example = pd.DataFrame([{
    "Distance_km": 5.2,
    "Delivery_person_Age": 28,
    "Delivery_person_Ratings": 4.5,
    "multiple_deliveries": 1,
    "Vehicle_condition": 2,
    "Order_Hour": 19,
    "Prep_time_min": 10,
    "Weather_conditions": "Fog",
    "Road_traffic_density": "High",
    "Type_of_order": "Meal",
    "Type_of_vehicle": "motorcycle",
    "Festival": "No",
    "City": "Metropolitian"
}])

predicted_minutes = best_pipe.predict(example)[0]
print(f"Predicted delivery time: {predicted_minutes:.1f} minutes")


## 8. Summary of Findings

Fill this section in with the numbers you see when you run the notebook -- typical patterns in this kind of dataset include:

- **Performance:** higher delivery-person ratings correlate with shorter delivery times; poor vehicle condition adds meaningful delay.
- **Route optimization:** delivery time grows non-linearly with traffic density (Jam >> Low) even at similar distances -- prioritize traffic-aware dispatch/routing over pure distance-based routing.
- **Customer experience:** adverse weather (Fog/Stormy/Sandstorms) and festival days both increase delivery time -- proactively communicate adjusted ETAs during these conditions.
- **Operations:** order volume peaks at specific hours (e.g. dinner rush) -- align rider shift scheduling accordingly; trips with more bundled deliveries take longer per order.
- **Predictive model:** tree-based models (Random Forest / Gradient Boosting) typically outperform plain linear regression here, with distance, traffic density, and weather as the top predictors of delivery time.
